In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Load cleaned dataset
df = pd.read_csv("data/processed/cleaned_clients.csv")

# Feature engineering
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

df["AvgMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["ServiceCount"] = (
    df[service_columns]
    .replace({
        "Yes": 1,
        "No": 0,
        "No internet service": 0,
        "No phone service": 0
    })
    .sum(axis=1)
)

# Separate features and target
X = df.drop(columns=["Churn", "customerID"])

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Identify feature types
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Setup completed successfully!")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

In [2]:
!pip install xgboost==3.4.1

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

xgb_validation_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=5174 / 1869,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_validation_model.fit(X_fit, y_fit)

val_probabilities = xgb_validation_model.predict_proba(X_val)[:, 1]

print("Validation samples:", len(X_val))
print("Validation churners:", int(y_val.sum()))

In [11]:
import xgboost
import sklearn
import numpy
import scipy
import joblib
import dill
import sys

print("Python:", sys.version)
print("XGBoost:", xgboost.__version__)
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Joblib:", joblib.__version__)
print("Dill:", dill.__version__)

In [4]:
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:

    val_predictions = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, val_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_val, val_predictions, zero_division=0
        ),
        "Customers Flagged": int(val_predictions.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(4)

In [5]:
selected_threshold = 0.40

final_xgb_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=5174 / 1869,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

# Train on the complete training set
final_xgb_model.fit(X_train, y_train)

print("Final XGBoost model trained!")

In [6]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Predict probabilities
y_test_proba = final_xgb_model.predict_proba(X_test)[:, 1]

# Apply selected threshold
y_test_pred = (
    y_test_proba >= selected_threshold
).astype(int)

print("Classification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["No Churn", "Churn"],
    digits=4
))

In [7]:
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    display_labels=["No Churn", "Churn"],
    cmap="Blues"
)

plt.title("XGBoost - Threshold 0.40")
plt.tight_layout()
plt.show()

In [8]:
print("Threshold:", selected_threshold)

print("Precision:",
      precision_score(y_test, y_test_pred))

print("Recall:",
      recall_score(y_test, y_test_pred))

print("F1 Score:",
      f1_score(y_test, y_test_pred))

print("ROC-AUC:",
      roc_auc_score(y_test, y_test_proba))

print("PR-AUC:",
      average_precision_score(y_test, y_test_proba))

In [9]:
%pip install shap

In [10]:
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [11]:
xgb_model = final_xgb_model.named_steps["model"]

fitted_preprocessor = (
    final_xgb_model.named_steps["preprocessor"]
)

In [12]:
X_test_transformed = fitted_preprocessor.transform(X_test)

feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

X_test_transformed_df = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=X_test.index
)

print(X_test_transformed_df.shape)

In [13]:
explainer = shap.TreeExplainer(xgb_model)

shap_values = explainer(
    X_test_transformed_df
)

print("SHAP calculation completed!")

In [14]:
shap.plots.bar(shap_values, max_display=15)

In [15]:
shap.plots.beeswarm(
    shap_values,
    max_display=15
)

In [16]:
# Select one customer from the test dataset
customer_index = 0

# Get that customer's SHAP explanation
customer_shap = shap_values[customer_index]

# Display the waterfall plot
shap.plots.waterfall(
    customer_shap,
    max_display=15
)

In [17]:
customer = X_test.iloc[customer_index]

print("Customer details:")
display(customer.to_frame(name="Value"))

print("\nActual churn status:",
      y_test.iloc[customer_index])

print("\nPredicted churn probability:",
      final_xgb_model.predict_proba(
          X_test.iloc[[customer_index]]
      )[0][1])

In [18]:
def explain_customer(customer_index, top_n=10):

    # Original customer information
    customer = X_test.iloc[customer_index]

    # Prediction probability
    probability = final_xgb_model.predict_proba(
        X_test.iloc[[customer_index]]
    )[0][1]

    prediction = int(probability >= selected_threshold)

    # Individual SHAP values
    customer_values = shap_values.values[customer_index]

    explanation = pd.DataFrame({
        "Feature": feature_names,
        "Transformed Value": X_test_transformed_df.iloc[
            customer_index
        ].values,
        "SHAP Value": customer_values
    })

    # Sort by absolute impact
    explanation["Impact"] = explanation[
        "SHAP Value"
    ].abs()

    explanation = explanation.sort_values(
        "Impact",
        ascending=False
    )

    print("Customer Index:", customer_index)
    print("Churn Probability:", round(probability * 100, 2), "%")
    print("Prediction:", "Churn" if prediction else "No Churn")

    print("\nTop factors influencing prediction:")

    display(
        explanation[
            ["Feature", "SHAP Value"]
        ].head(top_n)
    )

    return explanation, customer

In [19]:
customer_explanation, customer_details = explain_customer(
    customer_index=0,
    top_n=10
)

In [20]:
import os
import joblib

# Create models directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save model and selected threshold together
model_bundle = {
    "model": final_xgb_model,
    "threshold": selected_threshold
}

joblib.dump(
    model_bundle,
    "../models/churn_model_bundle.joblib"
)

print("Model saved successfully!")

In [21]:
# Inspect existing SHAP-related variables

shap_variables = [
    name for name in globals()
    if "shap" in name.lower()
    or "explainer" in name.lower()
]

print("SHAP-related variables:")
for name in shap_variables:
    print(name)

In [22]:
# Inspect model-related variables

model_variables = [
    name for name in globals()
    if "model" in name.lower()
    or "xgb" in name.lower()
]

print("Model-related variables:")
for name in model_variables:
    print(name)

In [23]:
print("Explainer type:", type(explainer))
print("SHAP values type:", type(shap_values))
print("Customer SHAP type:", type(customer_shap))

print("\nSHAP values shape:")
print(getattr(shap_values, "shape", "No shape attribute"))

print("\nCustomer SHAP shape:")
print(getattr(customer_shap, "shape", "No shape attribute"))

print("\nFinal model type:")
print(type(final_xgb_model))

print("\nModel bundle:")
print(type(model_bundle))

In [24]:
print(customer_shap)

In [25]:
if hasattr(customer_shap, "values"):
    print("Values shape:", customer_shap.values.shape)

if hasattr(customer_shap, "base_values"):
    print("Base values:", customer_shap.base_values)

if hasattr(customer_shap, "feature_names"):
    print("Feature names:", customer_shap.feature_names)

In [26]:
print("Model bundle contents:")

if isinstance(model_bundle, dict):
    for key, value in model_bundle.items():
        print(key, ":", type(value))
else:
    print(model_bundle)

In [27]:
print("MODEL BUNDLE KEYS")
print("-----------------")

for key, value in model_bundle.items():
    print(f"\nKey: {key}")
    print(f"Type: {type(value)}")

    if hasattr(value, "shape"):
        print("Shape:", value.shape)

In [28]:
print("MODEL PIPELINE STEPS")

for name, step in final_xgb_model.named_steps.items():
    print(name, ":", type(step))

In [29]:
import pandas as pd

customer_contributions = pd.DataFrame({
    "Feature": customer_shap.feature_names,
    "SHAP_Value": customer_shap.values
})

customer_contributions["Absolute_Impact"] = (
    customer_contributions["SHAP_Value"].abs()
)

display(
    customer_contributions
    .sort_values("Absolute_Impact", ascending=False)
    .head(10)
)

In [30]:
print("Model bundle keys:", model_bundle.keys())

print("\nModel type:", type(model_bundle["model"]))
print("Threshold:", model_bundle["threshold"])

print("\nPipeline steps:")
print(model_bundle["model"].named_steps)

In [31]:
import os
import joblib

os.makedirs("models", exist_ok=True)

model_path = "models/churn_model_bundle.joblib"

joblib.dump(model_bundle, model_path)

print("Model saved successfully!")
print("Location:", model_path)

In [32]:
import joblib

loaded_bundle = joblib.load(
    "models/churn_model_bundle.joblib"
)

loaded_model = loaded_bundle["model"]
loaded_threshold = loaded_bundle["threshold"]

print("Loaded model:", type(loaded_model))
print("Loaded threshold:", loaded_threshold)

assert isinstance(loaded_bundle, dict)
assert "model" in loaded_bundle
assert "threshold" in loaded_bundle

print("\nMODEL PACKAGING TEST PASSED!")

In [33]:
import numpy as np

original_probabilities = final_xgb_model.predict_proba(X_test)[:, 1]

loaded_probabilities = loaded_model.predict_proba(X_test)[:, 1]

difference = np.max(
    np.abs(original_probabilities - loaded_probabilities)
)

print("Maximum prediction difference:", difference)

assert np.allclose(
    original_probabilities,
    loaded_probabilities,
    atol=1e-7
)

print("PREDICTION CONSISTENCY TEST PASSED!")

In [34]:
# Inspect the saved model's expected input schema

loaded_model = loaded_bundle["model"]

expected_features = list(loaded_model.feature_names_in_)

print("Expected input feature count:", len(expected_features))

print("\nExpected input features:")
for i, feature in enumerate(expected_features, 1):
    print(f"{i}. {feature}")

print("\nPipeline components:")
for name, component in loaded_model.named_steps.items():
    print(name, ":", type(component))

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=2cb50308-81e6-4757-a2d6-155b4c4296dc' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>